# ML-10 — Content Action Playbook

This notebook operationalizes model predictions into a human-in-the-loop **Content Action Playbook** for **Lane 2 (Refresh / Content Opportunity Scoring)**. We map model probabilities to actionable editorial archetypes, establish strict operational boundaries, define a mandatory No-Go list for human review, specify retrain triggers, and export final action queues and figures.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `skills/writing-honest-claims/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.

## 1. Ranked actions + reason codes

### Mapping Predicted Decay Probabilities to Actionable Archetypes
We pull development data from `FlyRank/internship-warehouse` via DuckDB, generate predictions using our trained Random Forest model, and map probabilities $P(\text{declining}) \in [0.0, 1.0]$ to explicit human action labels and reason codes:

* $P \ge 0.75 \to$ Action: **`IMMEDIATE_REFRESH_CRITICAL`** | Reason Code: **`STALE_VISIBLE_DECAY`**
* $0.50 \le P < 0.75 \to$ Action: **`MONITOR_AND_EXPAND`** | Reason Code: **`MODERATE_DECAY_RISK`**
* $P < 0.50 \to$ Action: **`MAINTAIN_CURRENT_STATE`** | Reason Code: **`STABLE_PERFORMANCE`**

In [1]:
import os
import sys
import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
# 0. Safe Authentication & Warehouse Loading
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    for env_p in ['.env', '../.env', '../../.env']:
        if os.path.exists(env_p):
            with open(env_p, 'r', encoding='utf-8') as ef:
                for eline in ef:
                    if eline.startswith('HF_TOKEN'):
                        HF_TOKEN = eline.split('=', 1)[1].strip().strip('"\'')
                        break
assert HF_TOKEN is not None, "Error: Store HF_TOKEN in environment or secrets."
from huggingface_hub import hf_hub_download
repo_id = "FlyRank/internship-warehouse"
dim_content_path = hf_hub_download(repo_id=repo_id, filename="dim_content.parquet", repo_type="dataset", token=HF_TOKEN)
dim_clients_path = hf_hub_download(repo_id=repo_id, filename="dim_clients.parquet", repo_type="dataset", token=HF_TOKEN)
sample_perf_path = hf_hub_download(repo_id=repo_id, filename="fact_content_daily_performance_sample.parquet", repo_type="dataset", token=HF_TOKEN)
con = duckdb.connect()
con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")
con.execute(f"CREATE VIEW dim_clients AS SELECT * FROM read_parquet('{dim_clients_path}')")
con.execute(f"CREATE VIEW fact_performance_dev AS SELECT * FROM read_parquet('{sample_perf_path}')")
query_dev = """
SELECT
    f.content_hash_id as content_id,
    f.client_hash_id as client_id,
    SUM(f.gsc_impressions) as impressions_30d,
    SUM(f.gsc_clicks) as clicks_30d,
    AVG(f.gsc_avg_position) as avg_position,
    MAX(c.word_count) as word_count,
    DATEDIFF('day', MAX(c.content_created_date), MAX(f.report_date)) as content_age_days,
    MAX(CAST(f.ga4_data_available AS INT)) as ga4_data_available,
    CASE WHEN SUM(f.gsc_clicks) < 5 THEN 1 ELSE 0 END as is_declining_label
FROM fact_performance_dev f
JOIN dim_content c ON f.content_hash_id = c.content_hash_id
WHERE f.report_date IS NOT NULL
GROUP BY f.content_hash_id, f.client_hash_id
"""
df = con.execute(query_dev).df()
X = pd.DataFrame({
    'avg_position': df['avg_position'].fillna(100.0),
    'word_count': df['word_count'].fillna(1000.0),
    'content_age_days': df['content_age_days'].fillna(180.0),
    'ga4_data_available': df['ga4_data_available'].fillna(0),
    'log_impressions': np.log1p(df['impressions_30d'].fillna(0)),
})
y = df['is_declining_label']
# Train Random Forest Classifier
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=df['client_id']))
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
# Predict probabilities over whole dataset
df['opportunity_score'] = rf_model.predict_proba(X)[:, 1]
# Apply Rule Mapping for Action Archetypes
conditions = [
    (df['opportunity_score'] >= 0.75),
    (df['opportunity_score'] >= 0.50) & (df['opportunity_score'] < 0.75),
    (df['opportunity_score'] < 0.50)
]
action_labels = ['IMMEDIATE_REFRESH_CRITICAL', 'MONITOR_AND_EXPAND', 'MAINTAIN_CURRENT_STATE']
reason_codes = ['STALE_VISIBLE_DECAY', 'MODERATE_DECAY_RISK', 'STABLE_PERFORMANCE']
df['action_label'] = np.select(conditions, action_labels, default='MAINTAIN_CURRENT_STATE')
df['reason_code'] = np.select(conditions, reason_codes, default='STABLE_PERFORMANCE')
print("=== ACTION PLAYBOOK QUEUE SUMMARY ===")
action_summary = df.groupby(['action_label', 'reason_code']).agg(
    n_count=('content_id', 'count'),
    mean_score=('opportunity_score', 'mean'),
    mean_impressions=('impressions_30d', 'mean'),
    mean_clicks=('clicks_30d', 'mean')
).reset_index()
print(action_summary.round(3).to_string(index=False))


=== ACTION PLAYBOOK QUEUE SUMMARY ===
              action_label         reason_code  n_count  mean_score  mean_impressions  mean_clicks
IMMEDIATE_REFRESH_CRITICAL STALE_VISIBLE_DECAY   368465       0.987            62.689        0.259
    MAINTAIN_CURRENT_STATE  STABLE_PERFORMANCE    29641       0.251          6108.774       35.867
        MONITOR_AND_EXPAND MODERATE_DECAY_RISK    11099       0.650          1083.517        4.560


## 2. Intended use and limits

### Scope Boundary & Operational System Limitations

#### 1. Intended Use Scope
This playbook serves strictly as an **offline decision-support framework** designed to rank candidate URLs for human editorial review. It optimizes manual reviewer bandwidth by surfacing high-probability refresh opportunities.

#### 2. Explicit System Limitations
1. **Real-time SERP Volatility**: The framework evaluates historical performance slices and cannot predict live Google core algorithm updates or intraday SERP feature shifts.
2. **Hashed Query Intent**: Query identifiers are anonymized hashes; the model evaluates numerical volume metrics without semantic intent text analysis.
3. **Observational Panel History**: Historical data represents an observational panel; observed performance associations do not establish direct causality.

In [2]:
print("=== INTENDED USE & BOUNDARY AUDIT ===")
limits = [
    {"Boundary": "Intended Use", "Status": "VALID", "Details": "Offline human decision-support prioritization"},
    {"Boundary": "Real-Time Automation Loop", "Status": "INVALID", "Details": "Cannot trigger automated CMS edits directly"},
    {"Boundary": "SERP Volatility Window", "Status": "LIMITATION", "Details": "Lagged 30-day window; misses live daily core updates"},
    {"Boundary": "Semantic Intent Text", "Status": "LIMITATION", "Details": "Uses anonymized query hashes; no NLP text intent"}
]
print(pd.DataFrame(limits).to_string(index=False))


=== INTENDED USE & BOUNDARY AUDIT ===
                 Boundary     Status                                              Details
             Intended Use      VALID        Offline human decision-support prioritization
Real-Time Automation Loop    INVALID          Cannot trigger automated CMS edits directly
   SERP Volatility Window LIMITATION Lagged 30-day window; misses live daily core updates
     Semantic Intent Text LIMITATION     Uses anonymized query hashes; no NLP text intent


## 3. Human review + the no-go list

### Human-in-the-Loop Review Guidelines & Mandatory No-Go Rules

#### Human Review Checklist
Before executing content edits on flagged URLs (`IMMEDIATE_REFRESH_CRITICAL`), human editors must verify:
1. **Sibling Cannibalization Check**: Ensure another high-performing URL on the same domain does not target the identical primary keyword.
2. **Domain Authority Baseline**: Confirm whether the traffic decline is isolated to the specific URL or reflects sitewide tracking/domain drops.

#### The Mandatory No-Go List (Never Automate)
1. **Evergreen Reference Glossaries**: High-authority educational reference pages with low CTR must never be updated automatically.
2. **Seasonal Volatile Pages**: Landing pages for seasonal campaigns (e.g. holiday sales) must not be flagged based on off-season impression drops.
3. **Top-Converting Commercial Pages**: High-revenue landing pages require multi-department editorial consensus before structural changes.

In [3]:
print("=== MANDATORY NO-GO LIST RULES ===")
no_go_rules = [
    {"No-Go Rule": "1. Evergreen Glossaries", "Restriction": "Never automate CMS edits; preserve static reference text."},
    {"No-Go Rule": "2. Seasonal Landing Pages", "Restriction": "Exclude off-season volume drops from decay flags."},
    {"No-Go Rule": "3. Top Commercial Converters", "Restriction": "Require multi-department editorial approval prior to refresh."}
]
print(pd.DataFrame(no_go_rules).to_string(index=False))


=== MANDATORY NO-GO LIST RULES ===
                  No-Go Rule                                                   Restriction
     1. Evergreen Glossaries     Never automate CMS edits; preserve static reference text.
   2. Seasonal Landing Pages             Exclude off-season volume drops from decay flags.
3. Top Commercial Converters Require multi-department editorial approval prior to refresh.


## 4. Monitoring / retrain triggers

### System Maintenance & Recalibration Engineering Policy

To prevent recommendation staleness, system maintainers must enforce the following triggers:
1. **Performance Degradation Trigger**: Retrain models and recalibrate probability thresholds if operational **Precision@50** drops by more than **15%** on fresh quarterly performance updates.
2. **Tracking Architecture Migration Trigger**: Immediate pipeline re-fitting is mandatory whenever tracking systems undergo major migrations (e.g. GA4 tag reconfiguration or GSC property consolidation).

In [4]:
print("=== ENGINEERING RETRAIN & MONITORING POLICY ===")
policy = [
    {"Trigger Type": "Precision@50 Degradation", "Threshold": "> 15% drop from baseline", "Required Action": "Full model retraining & threshold recalibration"},
    {"Trigger Type": "Tracking Architecture Change", "Threshold": "GA4 / GSC tag migration", "Required Action": "Pipeline re-fitting & schema validation"}
]
print(pd.DataFrame(policy).to_string(index=False))


=== ENGINEERING RETRAIN & MONITORING POLICY ===
                Trigger Type                Threshold                                 Required Action
    Precision@50 Degradation > 15% drop from baseline Full model retraining & threshold recalibration
Tracking Architecture Change  GA4 / GSC tag migration         Pipeline re-fitting & schema validation


## 5. Exports for the paper

### Exporting Action Queues and Artifact Figures
We export the final ranked action queue to `work/outputs/action_playbook_queue.csv` and generate a distribution dataset saved to `work/figures/action_score_distribution.csv`.

In [5]:
import os
# Ensure output and figure directories exist
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)
# 1. Export Ranked Action Queue CSV
export_cols = ['content_id', 'client_id', 'opportunity_score', 'action_label', 'reason_code', 'impressions_30d', 'clicks_30d', 'avg_position', 'content_age_days']
ranked_queue = df.sort_values(by='opportunity_score', ascending=False)[export_cols]
csv_path = "../outputs/action_playbook_queue.csv"
ranked_queue.to_csv(csv_path, index=False)
print(f"Ranked Action Queue exported cleanly to: {csv_path}")
print(f"Exported Queue Shape: {ranked_queue.shape[0]:,} rows x {ranked_queue.shape[1]} columns\n")
# 2. Save Distribution Figure CSV Artifact
counts, bin_edges = np.histogram(df['opportunity_score'], bins=20)
dist_df = pd.DataFrame({
    'bin_lower': bin_edges[:-1],
    'bin_upper': bin_edges[1:],
    'item_count': counts
})
fig_csv_path = "../figures/action_score_distribution.csv"
dist_df.to_csv(fig_csv_path, index=False)
print(f"Validation Distribution Figure Artifact saved to: {fig_csv_path}")


Ranked Action Queue exported cleanly to: ../outputs/action_playbook_queue.csv
Exported Queue Shape: 409,205 rows x 9 columns

Validation Distribution Figure Artifact saved to: ../figures/action_score_distribution.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.